<a href="https://colab.research.google.com/github/CarterKatz/homework_02_rnn_text_experiment.ipynb/blob/main/homework_02_rnn_text_experiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Homework 2 - RNNs with Pretrained Word Embeddings

Train one recurrent neural network using frozen pretrained word embeddings. Use the trained RNN for AG News classification, document similarity, and query-based retrieval.

Choose one framework: PyTorch or TensorFlow with Keras. You do not need to implement both. You may use a Simple RNN, GRU, or LSTM. A bidirectional model is allowed but not required. Transformers are outside the scope of this assignment.

Implementation details are intentionally open-ended. Briefly describe the choices you make.

## Learning goals

- Prepare token sequences for a recurrent neural network.
- Initialize an embedding layer with pretrained word vectors.
- Freeze the pretrained embedding layer.
- Train an RNN, GRU, or LSTM for text classification.
- Extract one vector for each document from the trained RNN.
- Use document vectors for similarity and search.
- Use a GPU when one is available.

# Using a GPU

Google Colab is recommended if you do not have access to a local GPU.

1. Open the notebook in Google Colab.
2. Open the Runtime menu.
3. Select Change runtime type.
4. Select a GPU hardware accelerator.
5. Restart the runtime if Colab requests it.
6. Run the device check before training.

The command below will fail or show no GPU if a GPU runtime is not active. Run only the device example for your selected framework. PyTorch users must move the model and every training batch to the selected device. Keras normally uses an available GPU automatically.

GPU reference: https://research.google.com/colaboratory/faq.html#gpu-availability

In [ ]:
# Optional GPU check
# !nvidia-smi

# PyTorch example
# import torch
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print("Selected device", device)

# TensorFlow example
# import tensorflow as tf
# print("Available GPUs", tf.config.list_physical_devices("GPU"))

## Package setup

Run only the commands needed by your environment. Google Colab usually includes PyTorch and TensorFlow. Gensim may need to be installed. Do not import both deep-learning frameworks unless you want to.

In [ ]:
# %pip install gensim
# %pip install torch
# %pip install tensorflow

# The Experiment

1. Tokenize the training documents.
2. Build a vocabulary from the training text.
3. Convert documents into padded token sequences.
4. Load pretrained word vectors.
5. Initialize a frozen embedding layer.
6. Train a recurrent classifier.
7. Extract document vectors from the trained RNN.
8. Use the vectors for classification, similarity, and search.

The implementation details are open-ended. Record the choices you make.

# Load AG News

The source files have no header and contain label, title, and description. Labels 1 through 4 mean World, Sports, Business, and Sci/Tech. The official test data must remain separate from training data. The code below downloads both files and selects reproducible balanced subsets. You may reduce the training sample to 2,000 documents per category if runtime or memory is limited. Report the sample size you use.

In [ ]:
import pandas as pd

TRAIN_URL = (
    "https://raw.githubusercontent.com/mhjabreel/"
    "CharCnn_Keras/master/data/ag_news_csv/train.csv"
)
TEST_URL = (
    "https://raw.githubusercontent.com/mhjabreel/"
    "CharCnn_Keras/master/data/ag_news_csv/test.csv"
)
LABEL_NAMES = {1: "World", 2: "Sports", 3: "Business", 4: "Sci/Tech"}
RANDOM_STATE = 42

def load_ag_news(url):
    frame = pd.read_csv(url, header=None, names=["label", "title", "description"])
    frame["text"] = frame["title"].fillna("") + " " + frame["description"].fillna("")
    frame["label_name"] = frame["label"].map(LABEL_NAMES)
    frame["label_index"] = frame["label"] - 1
    return frame

train_all = load_ag_news(TRAIN_URL)
test_all = load_ag_news(TEST_URL)
train_per_category = 5000
test_per_category = 1000

train_df = (train_all.groupby("label", group_keys=False)
            .sample(n=train_per_category, random_state=RANDOM_STATE)
            .sample(frac=1, random_state=RANDOM_STATE)
            .reset_index(drop=True))
test_df = (test_all.groupby("label", group_keys=False)
           .sample(n=test_per_category, random_state=RANDOM_STATE)
           .sample(frac=1, random_state=RANDOM_STATE)
           .reset_index(drop=True))

train_df["doc_id"] = [f"train-{i:05d}" for i in range(len(train_df))]
test_df["doc_id"] = [f"test-{i:05d}" for i in range(len(test_df))]

print("Training shape", train_df.shape)
print("Test shape", test_df.shape)
print("Training counts\n", train_df["label_name"].value_counts())
print("Test counts\n", test_df["label_name"].value_counts())
display(train_df[["label_name", "text"]].head(4))

# Prepare Token Sequences

Build the vocabulary from training text only. Lowercase and tokenize each document. Reserve indices for padding and unknown words. Truncate long sequences, pad short sequences, and apply the same vocabulary to test documents and search queries. A simple regex tokenizer is acceptable. A framework tokenizer is also acceptable if you explain it. Packed sequences are optional research for PyTorch students.

Recommended settings:

`MAX_VOCAB_SIZE = 20000`, `MAX_SEQUENCE_LENGTH = 60`, `PAD_INDEX = 0`, `UNK_INDEX = 1`, and `RANDOM_STATE = 42`.

References: [PyTorch padding](https://pytorch.org/docs/stable/generated/torch.nn.utils.rnn.pad_sequence.html), [PyTorch packed sequences](https://pytorch.org/docs/stable/generated/torch.nn.utils.rnn.pack_padded_sequence.html), [Keras TextVectorization](https://www.tensorflow.org/api_docs/python/tf/keras/layers/TextVectorization), and [Keras padding](https://www.tensorflow.org/api_docs/python/tf/keras/utils/pad_sequences).

In [ ]:
import re

MAX_VOCAB_SIZE = 20000
MAX_SEQUENCE_LENGTH = 60
PAD_INDEX = 0
UNK_INDEX = 1
TOKEN_PATTERN = r"\b[a-zA-Z][a-zA-Z']*\b"

def tokenize(text):
    return re.findall(TOKEN_PATTERN, text.lower())

train_tokens = [tokenize(text) for text in train_df["text"]]
test_tokens = [tokenize(text) for text in test_df["text"]]

# TODO: build a training-only vocabulary, integer sequences, truncation, and padding.
# Keep PAD_INDEX and UNK_INDEX reserved, then encode test_tokens with the same mapping.
# Name your padded arrays train_sequences and test_sequences.

# Pretrained Word Embeddings

GloVe is a pretrained static word-embedding model. It serves the same role as pretrained Word2Vec vectors in this experiment. Do not use the large Google News Word2Vec model. The embedding layer must be initialized from GloVe and remain frozen during training.

In [ ]:
import gensim.downloader as api

word_vectors = api.load("glove-wiki-gigaword-50")
print("Embedding dimension", word_vectors.vector_size)

# Build the Embedding Matrix

Create a NumPy matrix with one row per vocabulary item and one column per embedding dimension. Use a pretrained vector when available. Use zeros for the padding row. Use zeros or a small random vector for unknown words. Do not build the vocabulary from test data.

Report the vocabulary size, embedding dimension, number of vocabulary words with pretrained vectors, and percentage covered by pretrained vectors.

In [ ]:
# TODO: build embedding_matrix after you create word_to_index.
# It should have shape (vocabulary_size, word_vectors.vector_size).
# Verify that the PAD_INDEX row is zero and count pretrained hits.
# Report vocabulary size, embedding dimension, hit count, and hit percentage.

## Embedding layer hints

These are short examples, not complete solutions. Verify and report that the embedding layer is frozen.

In [ ]:
# PyTorch example
# embedding_layer = torch.nn.Embedding.from_pretrained(
#     embedding_matrix, freeze=True, padding_idx=PAD_INDEX
# )

# TensorFlow with Keras example
# embedding_layer = tf.keras.layers.Embedding(
#     input_dim=vocabulary_size,
#     output_dim=embedding_dimension,
#     weights=[embedding_matrix],
#     trainable=False,
#     mask_zero=True
# )

# Build and Train a Recurrent Classifier

Your model should have this structure: token indices, frozen pretrained embedding layer, RNN, GRU, or LSTM, document vector, and a four-class output layer. Begin with one recurrent layer, hidden size 64 to 128, batch size 64 to 256, three to eight epochs, and Adam. These are suggestions. Use cross-entropy classification loss. The recurrent and classification layers must be trainable.

Documentation: [PyTorch Embedding](https://pytorch.org/docs/stable/generated/torch.nn.Embedding.html), [RNN](https://pytorch.org/docs/stable/generated/torch.nn.RNN.html), [GRU](https://pytorch.org/docs/stable/generated/torch.nn.GRU.html), [LSTM](https://pytorch.org/docs/stable/generated/torch.nn.LSTM.html), [CrossEntropyLoss](https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html), [Keras Embedding](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Embedding), [SimpleRNN](https://www.tensorflow.org/api_docs/python/tf/keras/layers/SimpleRNN), [GRU](https://www.tensorflow.org/api_docs/python/tf/keras/layers/GRU), and [LSTM](https://www.tensorflow.org/api_docs/python/tf/keras/layers/LSTM).

In [ ]:
# TODO: choose PyTorch or TensorFlow with Keras and build one recurrent classifier.
# Do not implement both frameworks. Do not use transformers.
# Create training batches, train for your chosen number of epochs, and record loss per epoch.
# Use the selected device and move model and batches there when using PyTorch.
# Plot training loss by epoch.
# Report framework, device, recurrent layer, vocabulary size, model summary,
# trainable parameter count, each epoch's loss, training time, and frozen status.

# Task 1 - News Topic Classification

Evaluate the trained model on the official balanced test subset. Display test accuracy, a confusion matrix, five example predictions, and at least three incorrect predictions. Each prediction must include text, true label, and predicted label. Use World, Sports, Business, and Sci/Tech as category names.

In [ ]:
# TODO: evaluate on test_sequences and test_df.
# Display accuracy, a confusion matrix, five predictions, and at least three mistakes.

### Short response

Describe one pattern you noticed in the model's mistakes.

TODO

# Extract Document Vectors

Modify or reuse the model so it returns the internal document vector before the final classification layer. Suitable choices include the final RNN, GRU, or LSTM hidden state. For an LSTM, use the hidden state rather than the cell state. For a bidirectional model, you may concatenate the final forward and backward states. Extract one vector for every test document and print the final matrix shape. It should have the form number of test documents by document-vector dimension. Do not create sequence-model vectors with a second model.

In [ ]:
# TODO: encode every test document with the trained RNN.
# Store the result in test_document_vectors and print its shape.

# Task 2 - Document Similarity

Select two test documents. Encode them with the trained RNN and compare each selected vector with all test-document vectors using cosine similarity. Exclude the query document itself and display the five most similar documents for each selection.

Each result table must contain document index, category, similarity score, and text. Reference: https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise.cosine_similarity.html

In [ ]:
# TODO: choose two test-document indices and compute cosine similarity.
# Exclude each query document from its own result and display five rows per query.

### Short response

Did the nearest documents share a category, vocabulary, or both?

TODO

# Task 3 - Query-Based Search

A new query must pass through the same tokenizer, vocabulary, unknown-word handling, sequence length, and trained RNN encoder as the documents. Encode each query and compare it with all test-document vectors using cosine similarity. Add one query of your own. Do not calculate retrieval metrics.

In [ ]:
queries = [
    "the team won the championship match",
    "technology company releases a new computer",
    "stocks rise after strong company earnings",
    "leaders meet to discuss the international conflict",
    # TODO: add one query of your own
]

# TODO: encode every query with the same pipeline used for test documents.
# For each query, display five results with rank, category, similarity score, and text.

### Short response

Which query worked best, and which query produced the weakest results?

TODO

# Final Results

Complete this compact summary table.

In [ ]:
# TODO: complete one row with your experiment results.
summary_columns = [
    "framework", "device", "recurrent layer", "hidden size",
    "number of recurrent layers", "bidirectional or unidirectional",
    "training sample size", "test sample size", "vocabulary size",
    "embedding dimension", "embedding layer frozen", "batch size",
    "number of epochs", "test accuracy", "training time"
]
summary_columns

### Final response

In a short paragraph, explain what the frozen word embeddings contributed and what the recurrent layer learned.

TODO

# Minimum required outputs

- [ ] AG News was downloaded and sampled reproducibly
- [ ] The vocabulary was built from training text only
- [ ] The embedding matrix used pretrained GloVe vectors
- [ ] The embedding layer remained frozen
- [ ] An RNN, GRU, or LSTM was trained
- [ ] Training loss was plotted
- [ ] Test accuracy and a confusion matrix were displayed
- [ ] Incorrect classifications were inspected
- [ ] RNN document vectors were extracted
- [ ] Similar documents were returned for two test documents
- [ ] Search results were returned for five text queries
- [ ] The final summary table and short responses were completed
- [ ] The notebook runs from top to bottom

Keep the assignment focused on one recurrent model in one framework. Do not add transformers, attention, language-model training, next-word prediction, trainable pretrained embeddings, sequence-to-sequence models, t-SNE, UMAP, precision at K, hyperparameter search, multiple required architectures, or a PyTorch and TensorFlow comparison.